## Manually Review like not_like tsv

Get all ytmusic entries, get like entries and aget not like entries, create artist - track key and move all matching likes and not like to respective playlist


In [1]:
import os
import sys
import pandas as pd

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
    
from ytmusic_library import YTMusicPlaylists

HEADER_FILE = '../oauth.json'
BACKUP_DIR = '../playlists/'
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=BACKUP_DIR)
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")


Using header file: ../oauth.json
Loaded 460 playlists


### Rerun get_like_not_like_tracks_to_review()

In [2]:
need_review = Y.get_like_not_like_tracks_to_review()
# need_review.to_csv(Y.need_rate_tsv, sep='\t', index=True)
# Note: now part of ytmusic backup


Loaded 38329 like, 6565 not like entries, and 151048 total tracks
Loaded 6565 not like entries, that have an entry in ALL_TRACKS
Keeping 6331 not like after remove LIKE
Processing LIKE tracks...
 Found 742 new tracks to LIKE
 Found 165 tracks to LIKE but already in NOT LIKE

Processing NOT LIKE tracks...
 Found 445 new tracks to NOT LIKE
 Found 202 tracks to NOT LIKE but they are already in LIKE

Top 10 playlists impacted:
decoded_list
z__thumbs_up_like    48
nan                  46
y_2005s_thumbs_up    45
Liked Music          33
indie                32
psych rock modern    30
y_2013_thumbs_up     28
y_2012_thumbs_up     27
y_2014_thumbs_up     24
electronic           21
Name: count, dtype: int64
Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
Reduced new LIKE from 742 to 480 entries after removing reviewed matches
Reduced new NOT_LIKE from 445 to 254 entries after removing reviewed matches
Reduced skip_not_like from 165 to 157 entries after removing reviewed matches

#### Previous versions:

```
v2.b
Loaded 38329 like, 6565 not like entries, and 151048 total tracks
Loaded 35452 like, 6565 not like entries, that have an entry in ALL_TRACKS
Processing LIKE tracks...
 Found 742 new tracks to LIKE
 Found 165 tracks to LIKE but already in NOT LIKE
Processing NOT LIKE tracks...
 Found 445 new tracks to NOT LIKE
 Found 202 tracks to NOT LIKE but they are already in LIKE
Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
Reduced new LIKE from 742 to 480 entries after removing reviewed matches
Reduced new NOT_LIKE from 445 to 254 entries after removing reviewed matches
Reduced skip_not_like from 165 to 157 entries after removing reviewed matches
Reduced skip_is_like from 202 to 201 entries after removing reviewed matches
Keeping 6331 not like after remove LIKE

v2.a (olderversions had dupes in Like)
Loaded 38329 like, 4996 not like entries, and 163918 total tracks
Keeping 4996 not like after remove LIKE
Saving _need_like tsv with 2868 entries
Saving _need_like_but_is_not_like tsv with 187 entries
Saving _need_not_like tsv with 4749 entries
Saving _need_not_like_but_is_like tsv with 259 entries

v1
Loaded 41461 like, 4806 not like entries, and 149082 total tracks
Keeping 4613 not like after remove LIKE
v0
Loaded 39646 like, 4531 not like entries, and 149039 total tracks
Keeping 4383 not like after remove LIKE
```

### Manual Step 1:

Import Y.need_rate_tsv into sheets and manually set 'manual_rating' category as:

`LIKE, NOT_LIKE, DISLIKE, INDIFFERENT`

In [28]:
like_df = pd.read_csv(Y.like_tsv, sep='\t', index_col=0)
not_like_df = pd.read_csv(Y.not_like_tsv, sep='\t', index_col=0)
manual_picks = pd.read_csv(Y.manual_rate_tsv, sep='\t', index_col=0).drop_duplicates(keep='last')

# use this to remove tracks already rated, then add ones that need rating to end or something
to_like = manual_picks.loc[manual_picks['manual_rating'] == 'LIKE']
to_not_like = manual_picks.loc[manual_picks['manual_rating'] == 'NOT_LIKE']
# Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
print(f'Loaded {len(manual_picks)} manually labeled entries, {len(to_like)} are LIKE, {len(to_not_like)} NOT_LIKE')

to_like_new_vids = frozenset(to_like.index) - frozenset(like_df.index)
to_not_like_new_vids = frozenset(to_not_like.index) - frozenset(not_like_df.index)
print(f'Filtered to new track video ids: {len(to_like_new_vids)} need LIKE, {len(to_not_like_new_vids)} need NOT_LIKE')


to_like_pl_id = Y.yt.create_playlist(
  title='_likes_new__manual_screen_tmp', video_ids=list(to_like_new_vids), privacy_status='PRIVATE')
is_actually_like = [t['videoId'] for t in Y.playlist_get_info(to_like_pl_id)['tracks'] if t['likeStatus'] == 'LIKE']
print(f'Removing {len(is_actually_like)} tracks from to_like because they currently are liked')
to_like_new_vids = to_like_new_vids - frozenset(is_actually_like)
print(f'Filtered to new track video ids: {len(to_like_new_vids)} need LIKE')
Y.yt.delete_playlist(to_like_pl_id)
to_like_pl_id = Y.yt.create_playlist(
  title='_likes_new__manual_screen', video_ids=list(to_like_new_vids), privacy_status='PRIVATE',
  description=f'{len(to_like_new_vids)} tracks that should be like')

to_not_like_pl_id = Y.yt.create_playlist(
  title='_not_likes_new__manual_screen', video_ids=list(to_not_like_new_vids), privacy_status='PRIVATE',
  description=f'{len(to_not_like_new_vids)} tracks that should be not like')
print(f'Created playlists for need like and need not like')


Loaded 8747 manually labeled entries, 3265 are LIKE, 4805 NOT_LIKE
Filtered to new track videoIdsL: 2858 need LIKE, 225 need NOT_LIKE


### Manual Step 2:

Look at to_not_like_pl_id on ytmusic:
* Should these all eb moved to not_like?
* If not remove from that playlist and fix: _ytmusic_new_like_and_not_like_manual_rated.tsv
* Once looks good add that playlist to zz not_Like 
Look at to_like_pl_id on ytmusic
* If any shouldnt be like,  remove from that playlist and fix: _ytmusic_new_like_and_not_like_manual_rated.tsv

Then go to next steps run code below to actually LIKE all 

In [44]:
# # Like the to_like playlist (can take a while, due to sleep time)
Y.playlist_rate_all_songs(Y.playlist_get_info(to_like_pl_id), 'LIKE', sleep_time=0.1,  verbose=False, skip_if_dislike=False)

# delete recently made playlists now that LIKE and NOT_LIKE established
Y.yt.delete_playlist(to_like_pl_id)
Y.yt.delete_playlist(to_not_like_pl_id)
print('Deleted to_like and to_not_like playlists')


Playlist _likes_new__manual_screen: Rated 569 of 783 tracks as LIKE
